# BlazingStar public federal-budget data — access tour

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abigailhaddad/blazingstar-data/blob/main/demo.ipynb)

[data.blazingstaranalytics.com](https://data.blazingstaranalytics.com/) is a free, **no-login, CC0 public-domain** mirror of the primary documents of federal spending and rulemaking, maintained by [BlazingStar Analytics](https://www.blazingstaranalytics.com/). It mirrors OMB MAX.gov, GovInfo/GPO, the Federal Register, and Regulations.gov.

Each dataset is a single `index.json` served from a Cloudflare CDN. **No API key, no auth.** Every record links back to its federal `source_url`; the execution data also carries a published SHA-256 hash for byte-level verification.

This notebook fetches each dataset into a pandas DataFrame, surfaces a weekly "what changed" signal and per-account execution rates, and verifies a file against its hash. It is unofficial and not affiliated with BlazingStar Analytics.

In [ ]:
# In Colab this installs deps; locally it's a no-op if already installed.
%pip install -q requests pandas openpyxl

In [ ]:
import hashlib
from io import BytesIO

import pandas as pd
import requests

# The CDN returns 403 to requests with no User-Agent, so use a session that sets one.
SESSION = requests.Session()
SESSION.headers.update({"User-Agent": "blazingstar-data-demo (https://github.com/abigailhaddad/blazingstar-data)"})


def get_json(url):
    """Fetch a JSON endpoint and return the parsed object."""
    r = SESSION.get(url, timeout=90)
    r.raise_for_status()
    return r.json()


def get_bytes(url):
    """Fetch a raw file (xlsx, pdf, mirrored json) as bytes."""
    r = SESSION.get(url, timeout=120)
    r.raise_for_status()
    return r.content


def load_apportionments(fy):
    """Return the cleaned, amount-bearing apportionment rows for a fiscal year.

    Each iteration is published twice (data_json + data_xlsx); only the
    data_json row carries the dollar amount. BUREAU-* rows are bureau-level
    rollups and are dropped so they don't double-count the accounts they
    aggregate.
    """
    idx = get_json(f"https://cdn.bzstr.co/sf132/fy{fy}/index.json")
    df = pd.DataFrame(idx["files"])
    df = df[(df["file_type"] == "data_json")
            & (~df["tafs"].str.startswith("BUREAU", na=False))].copy()
    df["amount"] = pd.to_numeric(df["total_approved_amount"], errors="coerce")
    df["iteration"] = pd.to_numeric(df["iteration"], errors="coerce")
    df["approved"] = pd.to_datetime(df["approval_timestamp"], utc=True, errors="coerce")
    return df

In [ ]:
# ---- Configuration: change these to point the notebook at other years ----
APPT_FY = 2026        # apportionments (SF-132) available FY2022–FY2026
APPENDIX_FY = 2027    # President's Budget Appendix (currently FY2027)
HISTORY_FYS = range(2022, 2027)   # FYs to compare in the multi-year rollup

## 1. Apportionments (SF-132)

OMB's apportionment schedules — the legal limits on how agencies may spend, keyed by **TAFS** (Treasury Account Symbol). One index per fiscal year, FY2022–FY2026, ~10k rows each. Updated nightly.

Endpoint: `https://cdn.bzstr.co/sf132/fy{fy}/index.json`

In [ ]:
appt_df = load_apportionments(APPT_FY)
print(f"FY{APPT_FY} apportionments — authoritative source: OMB MAX.gov (apportionment-public.max.gov)")
print("amount-bearing account rows:", len(appt_df), "| distinct TAFS:", appt_df["tafs"].nunique())

# Each row keeps its own source_url back to the OMB SF-132 it was mirrored from.
appt_df[["agency", "tafs", "account_name", "iteration", "amount", "approved", "source_url"]].head()

In [ ]:
# Current apportionment level for each account = its latest iteration.
latest = appt_df.sort_values("iteration").groupby(["agency", "tafs"], as_index=False).tail(1)

(
    latest.sort_values("amount", ascending=False)
    .loc[:, ["agency", "account_name", "tafs", "amount", "source_url"]]
    .head(10)
    .reset_index(drop=True)
)

In [ ]:
# Total apportioned (current iteration per account) across the available years.
history = []
for fy in HISTORY_FYS:
    fy_df = load_apportionments(fy)
    fy_latest = fy_df.sort_values("iteration").groupby(["agency", "tafs"]).tail(1)
    history.append({"fiscal_year": fy, "accounts": fy_latest["tafs"].nunique(),
                    "total_apportioned": fy_latest["amount"].sum()})
history_df = pd.DataFrame(history)
history_df.style.format({"total_apportioned": "${:,.0f}"})

## 2. What changed this week — re-apportionments

The apportionment file updates nightly, and any account (TAFS) can be apportioned more than once: each `iteration` is a fresh OMB action with its own `approval_timestamp`. Comparing an account's **latest iteration against the one before it** gives the dollar move, and filtering to recent approvals surfaces the week's activity — OMB releasing or pulling back budget authority before it shows up as awards in USAspending.

**Method (stated so it can be checked):**
- `data_json` rows only — they carry the dollar `total_approved_amount`.
- Unit = agency + TAFS; current = highest iteration, prior = the iteration immediately before it.
- `BUREAU-*` rollup rows excluded (they aggregate the individual accounts).
- The 7-day window is anchored to the **most recent approval in the file**, not your wall clock — so it stays meaningful whenever you run the notebook.
- Every row links to its OMB SF-132 `source_url` for verification.

In [ ]:
# Prior-iteration amount for each account, then the latest row per account.
appt_df = appt_df.sort_values(["agency", "tafs", "iteration"])
appt_df["prev_amount"] = appt_df.groupby(["agency", "tafs"])["amount"].shift(1)
current = appt_df.groupby(["agency", "tafs"]).tail(1).copy()
current["delta"] = current["amount"] - current["prev_amount"]

anchor = appt_df["approved"].max()
window_start = anchor - pd.Timedelta(days=7)
changed = current[(current["approved"] >= window_start)
                  & current["delta"].notna() & (current["delta"] != 0)].copy()
changed = changed.reindex(changed["delta"].abs().sort_values(ascending=False).index)

print(f"Window: {window_start.date()} → {anchor.date()} (anchored to latest approval in file)")
print(f"Accounts with a changed apportionment: {len(changed)}")
print(f"Net change: ${changed['delta'].sum():,.0f}   |   Gross moved: ${changed['delta'].abs().sum():,.0f}")

show = changed[["agency", "account_name", "tafs", "prev_amount", "amount", "delta", "source_url"]].head(15)
show.style.format({"prev_amount": "${:,.0f}", "amount": "${:,.0f}", "delta": "${:+,.0f}"})

In [ ]:
# Same week, rolled up by agency: net dollar move and how many accounts moved.
by_agency = (
    changed.groupby("agency")
    .agg(accounts=("tafs", "nunique"), net_delta=("delta", "sum"), gross_moved=("delta", lambda s: s.abs().sum()))
    .sort_values("gross_moved", ascending=False)
    .reset_index()
)
by_agency.style.format({"net_delta": "${:+,.0f}", "gross_moved": "${:,.0f}"})

## 3. Byte-level verification (SHA-256)

Each apportionment row carries a `mirror_url` (BlazingStar's copy) and a `hash_sha256`. Download the file and confirm the bytes match the published hash — this is the "don't take our word for it" guarantee. The `source_url` on the same row points to the original on OMB MAX.gov.

In [ ]:
row = appt_df[appt_df["mirror_url"].notna() & appt_df["hash_sha256"].notna()].iloc[0]
raw = get_bytes(row["mirror_url"])
computed = hashlib.sha256(raw).hexdigest()

print("file:    ", row["filename"])
print("bytes:   ", len(raw))
print("expected:", row["hash_sha256"])
print("computed:", computed)
print("MATCH:   ", computed == row["hash_sha256"])
print("source:  ", row["source_url"])

## 4. SF-133 Execution

OMB's monthly Report on Budget Execution — the raw spreadsheets, one per agency per period (submitter-identity columns blanked, financial data byte-identical to OMB). Browsable by agency and fiscal year.

Index: `https://liatris.blazingstaranalytics.com/sf133/index.json`

In [ ]:
sf133 = get_json("https://liatris.blazingstaranalytics.com/sf133/index.json")
print("source:    ", sf133["source"])
print("disclaimer:", sf133["disclaimer"][:140], "...")
print("agencies:", sf133["agency_count"], "| files:", sf133["file_count"])

# Each agency lists the fiscal years available; fiscal_year is on the year
# object, the latest file for that year is under "latest".
agency = sf133["agencies"][0]
year = agency["years"][0]
latest_sf133 = year["latest"]
print("\nagency: ", agency["human_name"])
print("period: ", latest_sf133["period_label"], "FY", year["fiscal_year"])
print("file:   ", latest_sf133["filename"], f"({latest_sf133['size']:,} bytes)")
print("source: ", latest_sf133["source_url"])

In [ ]:
# Download the monthly Excel file and load it straight into pandas.
xlsx = get_bytes(latest_sf133["url"])
print("verifies against hash_public:", hashlib.sha256(xlsx).hexdigest() == latest_sf133["hash_public"])

book = pd.ExcelFile(BytesIO(xlsx))
print("sheets:", book.sheet_names)

# "Raw Data" is the clean tabular sheet: one row per TAFS x SF-133 line.
sf133_df = book.parse("Raw Data")
print("shape:", sf133_df.shape, "| columns:", list(sf133_df.columns)[:8], "...")
sf133_df.head()

## 5. Execution rates (from SF-133)

How much of each account's budgetary resources has actually been **obligated** so far — the spending-pace signal. It's computable from a single SF-133 file (cumulative through the reporting period), using the standard line numbers:

- **Total budgetary resources** = line `1910` (the denominator; equals line `2500`).
- **Obligations incurred** = line `2190` (new obligations and upward adjustments).
- **Apportioned but not yet obligated** = lines `2201` + `2203` — money OMB has released that the agency hasn't committed yet.

The amounts are cumulative; we read the **latest reporting-period column** present in the file (here that's the column matching its period label). `pct_obligated = obligations / resources`.

In [ ]:
# SF-133 amounts live in period columns (AMT_<MONTH>); the cumulative figure we
# want is the latest populated period, in federal fiscal-calendar order.
MONTHS = ["OCT", "NOV", "DEC", "JAN", "FEB", "MAR", "APR", "MAY", "JUN", "JUL", "AUG", "SEP"]
month_cols = [f"AMT_{m}" for m in MONTHS if f"AMT_{m}" in sf133_df.columns]
current_col = [c for c in month_cols if sf133_df[c].fillna(0).abs().sum() > 0][-1]
print(f"reporting-period column: {current_col}  ({latest_sf133['period_label']})")

piv = sf133_df.pivot_table(index="TAFS", columns="LINENO", values=current_col, aggfunc="sum")


def line(n):
    """Series for an SF-133 line number, or NaNs if the line is absent."""
    return piv[n] if n in piv.columns else pd.Series(float("nan"), index=piv.index)


execu = pd.DataFrame({
    "resources": line(1910),
    "obligated": line(2190).fillna(0),
    "appt_unobligated": line(2201).fillna(0) + line(2203).fillna(0),
}).dropna(subset=["resources"])
execu = execu[execu["resources"] > 0]
execu["pct_obligated"] = execu["obligated"] / execu["resources"]

tot = execu[["resources", "obligated", "appt_unobligated"]].sum()
print(f"\n{agency['human_name']} — FY{year['fiscal_year']} through {latest_sf133['period_label']}")
print(f"  budgetary resources:      ${tot['resources']:,.0f}")
print(f"  obligated:                ${tot['obligated']:,.0f} ({tot['obligated'] / tot['resources']:.1%})")
print(f"  apportioned, unobligated: ${tot['appt_unobligated']:,.0f}")

show = execu.sort_values("resources", ascending=False).head(12)
show.style.format({"resources": "${:,.0f}", "obligated": "${:,.0f}",
                   "appt_unobligated": "${:,.0f}", "pct_obligated": "{:.1%}"})

## 6. President's Budget Appendix

The machine-readable President's Budget Appendix — Program & Financing schedules, object classification, and employment for every federal account, as structured JSON. One index per fiscal year listing 29 volumes.

Index: `https://cdn.bzstr.co/budget_appendix/{fy}/json/index.json`

In [ ]:
appendix = get_json(f"https://cdn.bzstr.co/budget_appendix/{APPENDIX_FY}/json/index.json")
print("FY:     ", appendix["fiscal_year"])
print("source: ", appendix["source"])
print("license:", appendix["license"])
print("totals: ", appendix["totals"])

# Each volume has a json_url with the per-account detail (P&F, object class, employment).
vol_df = pd.DataFrame(appendix["volumes"])
vol_df[["access_id", "department", "section_number", "source_url", "json_url"]].head(10)

## 7. CFR Redlines

Consequential Federal Register proposed rules that amend the CFR, redlined the morning they publish. Updated daily.

Index: `https://cdn.bzstr.co/redlines/index.json`

In [ ]:
redlines = get_json("https://cdn.bzstr.co/redlines/index.json")
redlines_df = pd.DataFrame(redlines)
print("rules:", len(redlines_df), "| columns:", list(redlines_df.columns))
# source_url = the Federal Register original; redline_url = BlazingStar's side-by-side.
redlines_df[["doc_number", "agency", "title", "comments_close_on", "source_url", "redline_url"]].head()

## 8. Spend Plans

The spend plans agencies file when OMB restricts their funds — public by court order, mirrored as PDFs and browsable by agency and year.

Index: `https://cdn.bzstr.co/spend-plans/index.json`

In [ ]:
plans = get_json("https://cdn.bzstr.co/spend-plans/index.json")
plans_df = pd.DataFrame(plans)
print("spend-plan PDFs:", len(plans_df))
# url = the mirrored PDF; source_url = where it was filed (MAX.gov).
plans_df[["agency", "bureau", "fiscal_year", "filename", "url", "source_url"]].head()

## The join key: TAFS

The **Treasury Account Symbol (TAFS)** ties these datasets together and out to [USAspending](https://www.usaspending.gov/): an apportionment (SF-132) sets the legal limit on a TAFS, the SF-133 reports execution against it, and awards in USAspending draw down from it — appropriation → apportionment → execution → award on one identifier. (Note the SF-132 and SF-133 mirrors format the TAFS string differently — `096-X-8862` vs `96-8862 /X` — so a cross-file join needs normalization first.)

## How to cite

The data is **CC0 1.0 (public domain)** — no attribution is legally required — but for any analysis you publish, cite the **primary federal source**, which travels with every record, and note the mirror you actually pulled from:

> *[Agency] [account / TAFS], [dataset], OMB MAX.gov / GovInfo / Federal Register, retrieved via BlazingStar Analytics' public mirror (data.blazingstaranalytics.com, CC0), [date].*

Every row in this notebook exposes a `source_url` pointing at the authoritative origin. For the apportionment and SF-133 figures specifically, the mirror publishes a **SHA-256** you can recompute (Section 3 above) — so a citation can assert byte-equivalence with OMB, not just "we downloaded a copy." When provenance matters, lead with the `source_url` and treat the BlazingStar mirror as the access path.